# Deep Learning Training: Skin Lesion Classification

## Overview

This notebook implements a comprehensive deep learning pipeline for multi-class skin lesion classification using the HAM10000 dataset. The approach leverages a custom Convolutional Neural Network (CNN) architecture optimized for dermatoscopic image analysis.

## Methodology

- **Dataset**: HAM10000 (Human Against Machine with 10,000 training images)
- **Classes**: 7 distinct skin lesion types (akiec, bcc, bkl, df, mel, nv, vasc)
- **Architecture**: Custom CNN with Batch Normalization and progressive feature extraction
- **Data Strategy**: Stratified train/validation split with aggressive augmentation for robustness
- **Training**: 15 epochs with categorical cross-entropy loss and Adam optimizer

## Key Features

- Comprehensive data preprocessing and organization
- Advanced data augmentation techniques
- Custom CNN architecture with batch normalization
- Robust evaluation metrics and visualizations
- Model checkpointing and persistence


## Import Libraries

In [ ]:
# Core libraries
import subprocess
import sys
import json

import os
import cv2
import shutil
import random
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
from PIL import Image

# Deep learning frameworks
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Activation, Flatten, 
    Dense, Dropout, BatchNormalization
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, 
    CSVLogger
)

# Machine learning utilities
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Visualization
from matplotlib import pyplot as plt
import matplotlib.patches as mpatches

import warnings
warnings.filterwarnings("ignore")

# Set random seeds for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Display TensorFlow version and GPU availability
print("=" * 60)
print("Environment Setup")
print("=" * 60)
print(f"TensorFlow version: {tf.__version__}")

# Configure GPU for TensorFlow - Force GPU usage
print("=" * 60)
print("GPU CONFIGURATION")
print("=" * 60)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Enable memory growth to avoid allocating all GPU memory at once
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        
        # Explicitly set GPU as the only visible device
        tf.config.set_visible_devices(gpus[0], 'GPU')
        
        # Verify GPU configuration
        logical_gpus = tf.config.list_logical_devices('GPU')
        print(f"✓ Physical GPUs: {len(gpus)}")
        print(f"✓ Logical GPUs: {len(logical_gpus)}")
        
        for i, gpu in enumerate(gpus):
            print(f"  GPU {i}: {gpu.name}")
            print(f"    Memory growth: Enabled")
        
        # Force GPU usage with a test computation
        print("\nTesting GPU computation...")
        with tf.device('/GPU:0'):
            test_a = tf.random.normal([1000, 1000])
            test_b = tf.random.normal([1000, 1000])
            test_c = tf.matmul(test_a, test_b)
            print(f"  ✓ GPU test successful!")
            print(f"  Test computation device: {test_c.device}")
            print(f"  ✓ TensorFlow WILL use GPU for training")
        
        # Set mixed precision for better GPU performance (optional but recommended)
        try:
            policy = tf.keras.mixed_precision.Policy('mixed_float16')
            tf.keras.mixed_precision.set_global_policy(policy)
            print(f"  ✓ Mixed precision enabled for better GPU performance")
        except:
            print(f"  ⚠️  Mixed precision not available (continuing anyway)")
            
    except RuntimeError as e:
        print(f"⚠️  GPU configuration error: {e}")
        print("  Training will fall back to CPU")
else:
    print("⚠️  No GPU devices found - training will use CPU")
    print("  Make sure GPU is enabled in Kaggle notebook settings")

print(f"\nEager Execution: {tf.executing_eagerly()}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print("=" * 60)
print("✓ All libraries imported successfully!")
print("=" * 60)

## 1. Dataset Loading and Exploration

### 1.1 Load Metadata

We begin by loading the HAM10000 metadata file, which contains image IDs and corresponding diagnostic labels. This dataset is a benchmark in dermatoscopic image classification.

In [ ]:
# Load metadata
METADATA_PATH = "/kaggle/input/skin-cancer-dataset/HAM10000_metadata.csv"
meta_data = pd.read_csv(METADATA_PATH)

print("=" * 60)
print("Dataset Overview")
print("=" * 60)
print(f"Total samples: {len(meta_data)}")
print(f"Columns: {list(meta_data.columns)}")
print(f"\nFirst few rows:")
print(meta_data.head())
print(f"\nDataset info:")
print(meta_data.info())
print(f"\nMissing values:")
print(meta_data.isnull().sum())

### 1.2 Class Distribution Analysis

Understanding class distribution is crucial for handling potential class imbalance and designing appropriate training strategies.

In [ ]:
# Display unique lesion types
print("=" * 60)
print("Lesion Type Analysis")
print("=" * 60)
print(f"\nUnique lesion types: {meta_data.dx.unique()}\n")

# Encode categorical labels to integers
encoder = LabelEncoder()
meta_data["dx_label"] = encoder.fit_transform(meta_data["dx"])

# Create mapping dictionary for reference
label_mapping = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))
print("Label Encoding Mapping:")
for lesion_type, label_id in label_mapping.items():
    print(f"  {lesion_type:6s} -> {label_id}")

# Analyze class distribution
class_counts = meta_data["dx"].value_counts().sort_index()
print("\n" + "=" * 60)
print("Class Distribution")
print("=" * 60)
for lesion_type in encoder.classes_:
    count = len(meta_data[meta_data["dx"] == lesion_type])
    percentage = (count / len(meta_data)) * 100
    print(f"{lesion_type:6s}: {count:5d} samples ({percentage:5.2f}%)")

# Visualize class distribution
plt.figure(figsize=(12, 6))
ax = class_counts.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Distribution of Skin Lesion Types', fontsize=14, fontweight='bold')
plt.xlabel('Lesion Type', fontsize=12)
plt.ylabel('Number of Samples', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, v in enumerate(class_counts.values):
    ax.text(i, v + 50, str(v), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n⚠️  Class imbalance detected - will use stratified sampling for train/validation split")

## 2. Data Preprocessing and Organization

### 2.1 Directory Structure Setup

We organize images into class-specific directories to facilitate efficient data loading and augmentation. This structure is compatible with Keras `flow_from_directory` and enables real-time augmentation during training.

In [ ]:
# Define paths
IMAGES_SOURCE_DIR = r"/kaggle/input/skin-cancer-dataset/Skin Cancer/Skin Cancer"
TRAIN_IMAGES_DIR = r"/kaggle/working/train/"
VALIDATION_IMAGES_DIR = r"/kaggle/working/validation/"

# Get directory names (encoded class labels)
dir_names = encoder.transform(encoder.classes_)

def create_directory_structure(base_path: str, class_labels: list) -> None:
    """
    Create directory structure for organized data storage.
    
    This function creates class-specific subdirectories to enable
    efficient data loading and maintain class separation.
    
    Args:
        base_path: Base directory path where subdirectories will be created
        class_labels: List of class label identifiers (integers)
    """
    for label in class_labels:
        label_dir = os.path.join(base_path, str(label))
        os.makedirs(label_dir, exist_ok=True)
    print(f"✓ Created directory structure at: {base_path}")

def organize_images_by_class(source_dir: str, target_dir: str, 
                             metadata: pd.DataFrame, encoder: LabelEncoder) -> dict:
    """
    Organize images into class-specific directories based on metadata.
    
    Args:
        source_dir: Directory containing original images
        target_dir: Target directory for organized images
        metadata: DataFrame with image_id and dx_label columns
        encoder: Fitted LabelEncoder for class mapping
        
    Returns:
        Dictionary with organization statistics
    """
    stats = {label: 0 for label in dir_names}
    processed = 0
    errors = 0
    
    print(f"\nOrganizing images from {source_dir}...")
    
    for image_file in os.scandir(source_dir):
        try:
            # Extract image ID (filename without extension)
            img_id = Path(image_file.name).stem
            
            # Get corresponding label from metadata
            if img_id in metadata['image_id'].values:
                label = str(metadata[metadata['image_id'] == img_id]['dx_label'].values[0])
                
                # Copy image to appropriate class directory
                target_path = os.path.join(target_dir, label, image_file.name)
                shutil.copy2(image_file.path, target_path)
                
                stats[int(label)] += 1
                processed += 1
                
                if processed % 500 == 0:
                    print(f"  Processed {processed} images...")
            else:
                errors += 1
        except Exception as e:
            errors += 1
            if errors <= 5:  # Print first few errors only
                print(f"  Warning: Could not process {image_file.name}: {e}")
    
    print(f"\n✓ Organization complete!")
    print(f"  Successfully processed: {processed} images")
    print(f"  Errors encountered: {errors}")
    
    return stats

# Create directory structure
create_directory_structure(TRAIN_IMAGES_DIR, dir_names)

# Organize images
organization_stats = organize_images_by_class(
    IMAGES_SOURCE_DIR, TRAIN_IMAGES_DIR, meta_data, encoder
)

# Display statistics
print("\n" + "=" * 60)
print("Organization Statistics")
print("=" * 60)
for label_id, count in organization_stats.items():
    lesion_type = encoder.inverse_transform([label_id])[0]
    print(f"Class {label_id} ({lesion_type:6s}): {count:5d} images")

### 2.2 Validation Set Creation

We create a stratified validation set by extracting 5% of images from each class. This ensures balanced representation across all lesion types and provides a reliable evaluation metric during training.

In [ ]:
# Check if validation directory already exists
VALIDATION_EXISTS = os.path.exists(VALIDATION_IMAGES_DIR) and any(
    os.listdir(os.path.join(VALIDATION_IMAGES_DIR, str(label))) 
    for label in dir_names 
    if os.path.exists(os.path.join(VALIDATION_IMAGES_DIR, str(label)))
)

if VALIDATION_EXISTS:
    print("⚠️  Validation directory already populated. Skipping validation set creation.")
    print("   To recreate, delete the validation directory first.")
else:
    print("Creating stratified validation set (5% per class)...")
    
    # Calculate validation set size per class (5% stratified)
    validation_split_ratio = 0.05
    validation_counts = {}
    
    # Count images per class
    for label_dir in os.scandir(TRAIN_IMAGES_DIR):
        if label_dir.is_dir():
            label_id = int(label_dir.name)
            image_count = len([f for f in os.scandir(label_dir) if f.is_file()])
            validation_size = max(1, int(image_count * validation_split_ratio))  # At least 1 image
            validation_counts[label_id] = validation_size
            
            lesion_type = encoder.inverse_transform([label_id])[0]
            print(f"  Class {label_id} ({lesion_type:6s}): {image_count:4d} total -> {validation_size:3d} for validation")
    
    # Create validation directory structure
    create_directory_structure(VALIDATION_IMAGES_DIR, dir_names)
    
    # Move validation samples
    moved_count = 0
    for label_dir in os.scandir(TRAIN_IMAGES_DIR):
        if label_dir.is_dir():
            label_id = int(label_dir.name)
            validation_size = validation_counts[label_id]
            
            # Get all images in this class directory
            all_images = [img.path for img in os.scandir(label_dir) if img.is_file()]
            
            # Randomly select validation samples
            random.shuffle(all_images)
            validation_images = all_images[:validation_size]
            
            # Move selected images to validation directory
            for img_path in validation_images:
                img_filename = os.path.basename(img_path)
                target_path = os.path.join(VALIDATION_IMAGES_DIR, str(label_id), img_filename)
                shutil.move(img_path, target_path)
                moved_count += 1
    
    print(f"\n✓ Validation set created: {moved_count} images moved to validation directory")
    
    # Verify final counts
    print("\nFinal dataset split:")
    for label_id in dir_names:
        train_path = os.path.join(TRAIN_IMAGES_DIR, str(label_id))
        val_path = os.path.join(VALIDATION_IMAGES_DIR, str(label_id))
        
        train_count = len([f for f in os.scandir(train_path) if f.is_file()]) if os.path.exists(train_path) else 0
        val_count = len([f for f in os.scandir(val_path) if f.is_file()]) if os.path.exists(val_path) else 0
        
        lesion_type = encoder.inverse_transform([label_id])[0]
        print(f"  {lesion_type:6s}: Train={train_count:4d}, Val={val_count:3d}, Total={train_count+val_count:4d}")

## 3. Data Augmentation and Generators

### 3.1 Augmentation Strategy

Data augmentation is critical for improving model generalization and handling class imbalance. We apply aggressive augmentation techniques including rotation, flipping, zooming, and brightness adjustments to increase dataset diversity.

In [ ]:
# Hyperparameters
IMG_SIZE = 250  # Input image dimensions (250x250)
BATCH_SIZE = 32
NUM_CLASSES = 7
VALIDATION_SPLIT = 0.1  # 10% of training data for validation during training

print("=" * 60)
print("Data Generator Configuration")
print("=" * 60)
print(f"Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Number of classes: {NUM_CLASSES}")
print(f"Training validation split: {VALIDATION_SPLIT*100}%")

# Training data generator with aggressive augmentation
# These augmentations help the model generalize better and handle:
# - Different orientations (rotation, flipping)
# - Scale variations (zoom)
# - Lighting conditions (brightness)
train_datagen = ImageDataGenerator(
    rescale=1./255,  # Normalize pixel values to [0, 1]
    zoom_range=0.3,  # Random zoom up to 30%
    rotation_range=90,  # Random rotation up to 90 degrees
    horizontal_flip=True,  # Random horizontal flip
    vertical_flip=True,  # Random vertical flip
    validation_split=VALIDATION_SPLIT  # Reserve 10% for validation
)

# Validation/test generators (no augmentation, only rescaling)
# This ensures we evaluate on original, unmodified images
val_test_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=VALIDATION_SPLIT
)

print("\nCreating data generators...")

# Training generator (with augmentation)
train_generator = train_datagen.flow_from_directory(
    directory=TRAIN_IMAGES_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=RANDOM_SEED
)

# Validation generator (during training, no augmentation)
val_generator = val_test_datagen.flow_from_directory(
    directory=TRAIN_IMAGES_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=RANDOM_SEED
)

# Development/test generator (final evaluation, no augmentation)
dev_generator = image_dataset_from_directory(
    VALIDATION_IMAGES_DIR,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=False
)

print("\n✓ Data generators created successfully!")
print(f"\nTraining samples: {train_generator.samples}")
print(f"Validation samples (during training): {val_generator.samples}")
print(f"Development samples: {len(list(dev_generator.file_paths)) if hasattr(dev_generator, 'file_paths') else 'N/A'}")

# Display class indices mapping
print("\nClass indices mapping:")
for class_name, class_idx in train_generator.class_indices.items():
    lesion_type = encoder.inverse_transform([int(class_name)])[0]
    print(f"  Index {class_idx}: {lesion_type} (label {class_name})")

## 4. Model Architecture

### 4.1 CNN Architecture Design

We design a custom CNN architecture optimized for dermatoscopic image classification. The architecture follows a progressive feature extraction approach:

- **Batch Normalization**: Stabilizes training and accelerates convergence
- **Convolutional Layers**: Extract hierarchical features (64 → 32 filters)
- **Max Pooling**: Reduces spatial dimensions and computational complexity
- **Dense Layers**: Perform final classification (64 → 32 → 7 neurons)

The architecture balances model capacity with computational efficiency, making it suitable for medical image classification tasks.


In [ ]:
def build_skin_lesion_cnn(input_shape=(250, 250, 3), num_classes=7):
    """
    Build a custom CNN for skin lesion classification.
    
    Architecture:
    - Input: 250x250x3 RGB images
    - Conv Block 1: 64 filters, BatchNorm, ReLU, MaxPool
    - Conv Block 2: 32 filters, BatchNorm, ReLU, MaxPool
    - Dense Block 1: 64 neurons, ReLU
    - Dense Block 2: 32 neurons, ReLU
    - Output: 7 neurons (softmax for multi-class classification)
    
    Args:
        input_shape: Tuple specifying input image dimensions (height, width, channels)
        num_classes: Number of output classes
        
    Returns:
        Compiled Keras model
    """
    model = Sequential(name="SkinLesionCNN")
    
    # Input normalization layer
    # BatchNormalization helps stabilize training and allows higher learning rates
    model.add(BatchNormalization(input_shape=input_shape, name='input_bn'))
    
    # Convolutional Block 1: Extract low-level features (edges, textures)
    model.add(Conv2D(64, kernel_size=(3, 3), padding='same', name='conv1'))
    model.add(Activation("relu", name='conv1_relu'))
    model.add(MaxPooling2D(pool_size=(2, 2), name='conv1_pool'))
    
    # Convolutional Block 2: Extract mid-level features (patterns, shapes)
    model.add(Conv2D(32, kernel_size=(3, 3), padding='same', name='conv2'))
    model.add(Activation("relu", name='conv2_relu'))
    model.add(MaxPooling2D(pool_size=(2, 2), name='conv2_pool'))
    
    # Flatten convolutional features for dense layers
    model.add(Flatten(name='flatten'))
    
    # Dense Block 1: High-level feature processing
    model.add(Dense(64, name='dense1'))
    model.add(Activation("relu", name='dense1_relu'))
    
    # Dense Block 2: Final feature refinement
    model.add(Dense(32, name='dense2'))
    model.add(Activation("relu", name='dense2_relu'))
    
    # Output layer: Multi-class classification
    model.add(Dense(num_classes, name='output'))
    model.add(Activation("softmax", name='output_softmax'))
    
    return model

# Build model
print("Building CNN architecture...")
model = build_skin_lesion_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES)

# Compile model
# Categorical cross-entropy is appropriate for multi-class classification
# Adam optimizer provides adaptive learning rates
model.compile(
    loss="categorical_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    metrics=["accuracy", keras.metrics.TopKCategoricalAccuracy(k=2, name='top2_accuracy')]
)

# Display model architecture
print("\n" + "=" * 60)
print("Model Architecture Summary")
print("=" * 60)
model.summary()

# Calculate total parameters
total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Non-trainable parameters: {total_params - trainable_params:,}")

# Visualize model architecture (if graphviz is available)
try:
    keras.utils.plot_model(model, to_file='/kaggle/working/model_architecture.png', 
                          show_shapes=True, show_layer_names=True, rankdir='TB')
    print("\n✓ Model architecture diagram saved to model_architecture.png")
except:
    print("\n⚠️  Could not generate model diagram (graphviz not available)")


## 5. Training Configuration

### 5.1 Callbacks and Training Strategy

We implement several callbacks to optimize training:
- **EarlyStopping**: Prevents overfitting by stopping when validation loss plateaus
- **ModelCheckpoint**: Saves best model weights during training
- **ReduceLROnPlateau**: Dynamically reduces learning rate for fine-tuning
- **CSVLogger**: Tracks training metrics for analysis


In [ ]:
# Training hyperparameters
EPOCHS = 15
INITIAL_LR = 0.001

# Setup callbacks for optimal training
callbacks = [
    # Early stopping: Stop training if validation loss doesn't improve
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1,
        mode='min'
    ),
    
    # Model checkpointing: Save best model weights
    ModelCheckpoint(
        filepath='/kaggle/working/best_model.keras',
        monitor='val_loss',
        save_best_only=True,
        save_weights_only=False,
        verbose=1,
        mode='min'
    ),
    
    # Learning rate reduction: Reduce LR when validation loss plateaus
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1,
        mode='min'
    ),
    
    # CSV logger: Save training history
    CSVLogger(
        filename='/kaggle/working/training_history.csv',
        separator=',',
        append=False
    )
]

print("=" * 60)
print("Training Configuration")
print("=" * 60)
print(f"Epochs: {EPOCHS}")
print(f"Initial learning rate: {INITIAL_LR}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {val_generator.samples}")
print(f"\nCallbacks configured:")
for cb in callbacks:
    print(f"  - {cb.__class__.__name__}")

# Add GPU verification callback
class GPUVerificationCallback(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        gpus = tf.config.list_logical_devices('GPU')
        print(f"\n{'='*60}")
        print("GPU VERIFICATION")
        print(f"{'='*60}")
        if gpus:
            print(f"✓ Training on GPU: {gpus[0].name}")
            # Check where model variables are placed
            for i, var in enumerate(self.model.trainable_variables[:3]):
                print(f"  Variable {i} device: {var.device}")
            if '/GPU' in str(self.model.trainable_variables[0].device):
                print("  ✓ Model is on GPU - Training will use GPU!")
            else:
                print("  ⚠️  Model variables not on GPU!")
        else:
            print("⚠️  WARNING: No GPU detected - training on CPU!")
        print(f"{'='*60}\n")

callbacks.append(GPUVerificationCallback())

print("\n" + "=" * 60)
print("Starting Training...")
print("=" * 60)


In [ ]:
# Train the model
# Note: fit_generator was deprecated and removed in TensorFlow 2.18.0
# The fit() method now handles generators directly
# Force GPU usage by wrapping in device context
print("\n" + "=" * 60)
print("TRAINING STARTED")
print("=" * 60)
print(f"Device placement: GPU (forced)")
print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {val_generator.samples}")
print(f"Epochs: {EPOCHS}")
print("=" * 60 + "\n")

# Ensure model is on GPU
with tf.device('/GPU:0'):
    history = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=EPOCHS,
        verbose=1,
        callbacks=callbacks
    )

print("\n" + "=" * 60)
print("Training Completed!")
print("=" * 60)

# Save final model
final_model_path = '/kaggle/working/final_model.keras'
model.save(final_model_path)
print(f"\n✓ Final model saved to: {final_model_path}")

# Display training summary
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
best_val_acc = max(history.history['val_accuracy'])

print(f"\nTraining Summary:")
print(f"  Final training accuracy: {final_train_acc:.4f}")
print(f"  Final validation accuracy: {final_val_acc:.4f}")
print(f"  Best validation accuracy: {best_val_acc:.4f}")


## 6. Model Evaluation and Analysis

### 6.1 Training History Visualization

We analyze the training process to understand model behavior, detect overfitting, and assess convergence.


In [ ]:
# Extract training history
metrics = history.history
epochs = range(1, len(metrics['loss']) + 1)

# Create comprehensive training visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Training History Analysis', fontsize=16, fontweight='bold')

# Plot 1: Loss
axes[0, 0].plot(epochs, metrics['loss'], 'b-o', label='Training Loss', linewidth=2, markersize=6)
axes[0, 0].plot(epochs, metrics['val_loss'], 'r-s', label='Validation Loss', linewidth=2, markersize=6)
axes[0, 0].set_title('Model Loss', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()
axes[0, 0].set_ylim(bottom=0)

# Plot 2: Accuracy
axes[0, 1].plot(epochs, metrics['accuracy'], 'b-o', label='Training Accuracy', linewidth=2, markersize=6)
axes[0, 1].plot(epochs, metrics['val_accuracy'], 'r-s', label='Validation Accuracy', linewidth=2, markersize=6)
axes[0, 1].set_title('Model Accuracy', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()
axes[0, 1].set_ylim([0, 1])

# Plot 3: Top-2 Accuracy (if available)
if 'top2_accuracy' in metrics:
    axes[1, 0].plot(epochs, metrics['top2_accuracy'], 'g-o', label='Training Top-2 Accuracy', linewidth=2, markersize=6)
    axes[1, 0].plot(epochs, metrics['val_top2_accuracy'], 'm-s', label='Validation Top-2 Accuracy', linewidth=2, markersize=6)
    axes[1, 0].set_title('Top-2 Accuracy', fontsize=12, fontweight='bold')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Top-2 Accuracy')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].legend()
    axes[1, 0].set_ylim([0, 1])
else:
    axes[1, 0].axis('off')

# Plot 4: Learning Rate (if available)
if 'lr' in metrics:
    axes[1, 1].plot(epochs, metrics['lr'], 'purple', linewidth=2, marker='o', markersize=6)
    axes[1, 1].set_title('Learning Rate Schedule', fontsize=12, fontweight='bold')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Learning Rate')
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].set_yscale('log')
else:
    axes[1, 1].axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/training_history.png', dpi=150, bbox_inches='tight')
plt.show()

# Print key metrics
print("\n" + "=" * 60)
print("Training Metrics Summary")
print("=" * 60)
print(f"Best Training Accuracy: {max(metrics['accuracy']):.4f} (Epoch {np.argmax(metrics['accuracy'])+1})")
print(f"Best Validation Accuracy: {max(metrics['val_accuracy']):.4f} (Epoch {np.argmax(metrics['val_accuracy'])+1})")
print(f"Final Training Loss: {metrics['loss'][-1]:.4f}")
print(f"Final Validation Loss: {metrics['val_loss'][-1]:.4f}")

# Check for overfitting
overfitting_gap = max(metrics['accuracy']) - max(metrics['val_accuracy'])
if overfitting_gap > 0.1:
    print(f"\n⚠️  Potential overfitting detected (gap: {overfitting_gap:.4f})")
else:
    print(f"\n✓ Model shows good generalization (gap: {overfitting_gap:.4f})")


### 6.2 Development Set Evaluation

We evaluate the model on the held-out development set to assess real-world performance and generate detailed classification metrics.


In [ ]:
# Load best model weights for evaluation
try:
    model.load_weights('/kaggle/working/best_model.keras')
    print("✓ Loaded best model weights for evaluation")
except:
    print("⚠️  Using final model weights (best weights not found)")

# Make predictions on development set
print("\nGenerating predictions on development set...")

# Convert dev_generator to numpy arrays for prediction
dev_images = []
dev_labels = []

for batch_images, batch_labels in dev_generator:
    dev_images.append(batch_images.numpy())
    dev_labels.append(batch_labels.numpy())
    if len(dev_images) * BATCH_SIZE >= 1000:  # Safety limit
        break

dev_images = np.concatenate(dev_images, axis=0)
dev_labels = np.concatenate(dev_labels, axis=0)

# Normalize images (if not already normalized)
if dev_images.max() > 1.0:
    dev_images = dev_images / 255.0

# Generate predictions
predictions = model.predict(dev_images, batch_size=BATCH_SIZE, verbose=1)
predicted_classes = np.argmax(predictions, axis=1)

print(f"\n✓ Predictions generated for {len(predicted_classes)} samples")


In [ ]:
# Extract true labels
true_classes = dev_labels.tolist()

# Ensure matching lengths
min_length = min(len(true_classes), len(predicted_classes))
true_classes = true_classes[:min_length]
predicted_classes = predicted_classes[:min_length]

# Generate comprehensive classification report
print("\n" + "=" * 60)
print("Classification Report")
print("=" * 60)
report = classification_report(
    true_classes, 
    predicted_classes,
    target_names=[encoder.inverse_transform([i])[0] for i in range(NUM_CLASSES)],
    digits=4
)
print(report)

# Calculate overall accuracy
overall_accuracy = np.mean(np.array(true_classes) == np.array(predicted_classes))
print(f"\nOverall Accuracy: {overall_accuracy:.4f} ({overall_accuracy*100:.2f}%)")

# Per-class accuracy
print("\n" + "=" * 60)
print("Per-Class Performance")
print("=" * 60)
for i in range(NUM_CLASSES):
    lesion_type = encoder.inverse_transform([i])[0]
    class_mask = np.array(true_classes) == i
    if np.sum(class_mask) > 0:
        class_accuracy = np.mean(np.array(predicted_classes)[class_mask] == i)
        class_count = np.sum(class_mask)
        print(f"{lesion_type:6s}: {class_accuracy:.4f} ({class_count} samples)")


In [ ]:
# Investigating F1 score.    
# Calculating confusion matrix
plt.title("Heatmap of devset prediction and actual label", fontsize = 13)
cm = confusion_matrix(true_class, predicted_class)
sns.heatmap(cm, cmap = "Reds", annot = True, fmt = "d");
plt.ylabel("True Class");
plt.xlabel("Predicted Class");


In [ ]:
# Generate confusion matrix
cm = confusion_matrix(true_classes, predicted_classes)

# Create enhanced confusion matrix visualization
fig, ax = plt.subplots(figsize=(10, 8))

# Normalize confusion matrix for better interpretation
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Create heatmap with both counts and percentages
class_names = [encoder.inverse_transform([i])[0] for i in range(NUM_CLASSES)]
sns.heatmap(cm_normalized, 
            annot=True, 
            fmt='.2%',
            cmap='Reds',
            xticklabels=class_names,
            yticklabels=class_names,
            cbar_kws={'label': 'Normalized Frequency'},
            ax=ax,
            linewidths=0.5,
            linecolor='gray')

ax.set_title('Normalized Confusion Matrix\n(Development Set)', 
              fontsize=14, fontweight='bold', pad=20)
ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix_normalized.png', dpi=150, bbox_inches='tight')
plt.show()

# Also create raw count confusion matrix
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names,
            cbar_kws={'label': 'Count'},
            ax=ax,
            linewidths=0.5,
            linecolor='gray')

ax.set_title('Confusion Matrix - Raw Counts\n(Development Set)', 
              fontsize=14, fontweight='bold', pad=20)
ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix_counts.png', dpi=150, bbox_inches='tight')
plt.show()

# Analyze confusion patterns
print("\n" + "=" * 60)
print("Confusion Analysis")
print("=" * 60)
for i in range(NUM_CLASSES):
    lesion_type = encoder.inverse_transform([i])[0]
    correct = cm[i, i]
    total = cm[i, :].sum()
    if total > 0:
        recall = correct / total
        print(f"{lesion_type:6s}: {correct}/{total} correct (Recall: {recall:.4f})")
        
        # Show most common misclassifications
        misclassifications = cm[i, :].copy()
        misclassifications[i] = 0  # Remove correct predictions
        if misclassifications.sum() > 0:
            most_confused = np.argmax(misclassifications)
            confused_type = encoder.inverse_transform([most_confused])[0]
            confused_count = misclassifications[most_confused]
            print(f"         Most confused with: {confused_type} ({confused_count} times)")


In [ ]:
# Save model artifacts
import json

# Save label encoder mapping
label_mapping = {
    'classes': encoder.classes_.tolist(),
    'label_to_id': {cls: int(encoder.transform([cls])[0]) for cls in encoder.classes_},
    'id_to_label': {int(encoder.transform([cls])[0]): cls for cls in encoder.classes_}
}

with open('/kaggle/working/label_mapping.json', 'w') as f:
    json.dump(label_mapping, f, indent=2)

# Save model configuration
model_config = {
    'input_shape': (IMG_SIZE, IMG_SIZE, 3),
    'num_classes': NUM_CLASSES,
    'batch_size': BATCH_SIZE,
    'architecture': 'Custom CNN',
    'training_epochs': EPOCHS,
    'final_accuracy': float(max(history.history['val_accuracy'])),
    'development_accuracy': float(overall_accuracy)
}

with open('/kaggle/working/model_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)

print("=" * 60)
print("Model Artifacts Saved")
print("=" * 60)
print("✓ Model weights: /kaggle/working/best_model.keras")
print("✓ Final model: /kaggle/working/final_model.keras")
print("✓ Label mapping: /kaggle/working/label_mapping.json")
print("✓ Model configuration: /kaggle/working/model_config.json")
print("✓ Training history: /kaggle/working/training_history.csv")
print("✓ Training plots: /kaggle/working/training_history.png")
print("✓ Confusion matrices: /kaggle/working/confusion_matrix_*.png")

print("\n" + "=" * 60)
print("Training Pipeline Complete!")
print("=" * 60)
print(f"\nModel Performance Summary:")
print(f"  Best Validation Accuracy: {max(history.history['val_accuracy']):.4f}")
print(f"  Development Set Accuracy: {overall_accuracy:.4f}")
print(f"  Total Parameters: {model.count_params():,}")
print(f"\nThe model is ready for deployment and inference!")
